In [1]:
# ============================================
# NOTEBOOK 1: EXPLORING STATSBOMB DATA
# StatsBomb provide free open data covering
# 60+ competitions including the Champions
# League, FA Women's Super League, La Liga
# and more — all free to use
# ============================================

from statsbombpy import sb
import pandas as pd
import numpy as np

# See all available competitions
competitions = sb.competitions()
print("Total competitions available:", len(competitions))
print("\nSample competitions:")
print(competitions[['competition_name', 
                     'season_name', 
                     'country_name']].head(20))

C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Total competitions available: 80

Sample competitions:
          competition_name season_name country_name
0            1. Bundesliga   2023/2024      Germany
1            1. Bundesliga   2015/2016      Germany
2   African Cup of Nations        2023       Africa
3         Champions League   2018/2019       Europe
4         Champions League   2017/2018       Europe
5         Champions League   2016/2017       Europe
6         Champions League   2015/2016       Europe
7         Champions League   2014/2015       Europe
8         Champions League   2013/2014       Europe
9         Champions League   2012/2013       Europe
10        Champions League   2011/2012       Europe
11        Champions League   2010/2011       Europe
12        Champions League   2009/2010       Europe
13        Champions League   2008/2009       Europe
14        Champions League   2006/2007       Europe
15        Champions League   2004/2005       Europe
16        Champions League   2003/2004       Europe
17       

In [2]:
# Print every competition available
print(competitions[['competition_id', 
                     'competition_name', 
                     'season_id',
                     'season_name']].to_string())

    competition_id         competition_name  season_id season_name
0                9            1. Bundesliga        281   2023/2024
1                9            1. Bundesliga         27   2015/2016
2             1267   African Cup of Nations        107        2023
3               16         Champions League          4   2018/2019
4               16         Champions League          1   2017/2018
5               16         Champions League          2   2016/2017
6               16         Champions League         27   2015/2016
7               16         Champions League         26   2014/2015
8               16         Champions League         25   2013/2014
9               16         Champions League         24   2012/2013
10              16         Champions League         23   2011/2012
11              16         Champions League         22   2010/2011
12              16         Champions League         21   2009/2010
13              16         Champions League         41   2008/

In [3]:
# Filter for La Liga specifically
la_liga = competitions[competitions['competition_name'] == 'La Liga']
print(la_liga[['competition_id', 'competition_name', 
               'season_id', 'season_name']])

    competition_id competition_name  season_id season_name
40              11          La Liga         90   2020/2021
41              11          La Liga         42   2019/2020
42              11          La Liga          4   2018/2019
43              11          La Liga          1   2017/2018
44              11          La Liga          2   2016/2017
45              11          La Liga         27   2015/2016
46              11          La Liga         26   2014/2015
47              11          La Liga         25   2013/2014
48              11          La Liga         24   2012/2013
49              11          La Liga         23   2011/2012
50              11          La Liga         22   2010/2011
51              11          La Liga         21   2009/2010
52              11          La Liga         41   2008/2009
53              11          La Liga         40   2007/2008
54              11          La Liga         39   2006/2007
55              11          La Liga         38   2005/20

In [4]:
# See all available La Liga seasons
print("La Liga Seasons Available:")
print(la_liga[['season_id', 'season_name']].to_string())

La Liga Seasons Available:
    season_id season_name
40         90   2020/2021
41         42   2019/2020
42          4   2018/2019
43          1   2017/2018
44          2   2016/2017
45         27   2015/2016
46         26   2014/2015
47         25   2013/2014
48         24   2012/2013
49         23   2011/2012
50         22   2010/2011
51         21   2009/2010
52         41   2008/2009
53         40   2007/2008
54         39   2006/2007
55         38   2005/2006
56         37   2004/2005
57        278   1973/1974


In [5]:
# ============================================
# LOAD LA LIGA 2020/2021 MATCHES
# competition_id = 11 (La Liga)
# season_id = 90 (2020/2021)
# ============================================

matches = sb.matches(competition_id=11, season_id=90)

print("Total matches:", len(matches))
print("\nColumns available:")
print(matches.columns.tolist())
print("\nFirst match:")
print(matches[['match_id', 'home_team', 
               'away_team', 'match_date']].head(5))

C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Total matches: 35

Columns available:
['match_id', 'match_date', 'kick_off', 'home_score', 'away_score', 'match_status', 'match_status_360', 'last_updated', 'last_updated_360', 'match_week', 'competition_id', 'competition_country_name', 'competition_name', 'competition', 'season_id', 'season', 'home_team_id', 'home_team', 'home_team_gender', 'home_team_group', 'home_team_country_id', 'home_team_country_name', 'away_team_id', 'away_team', 'away_team_gender', 'away_team_group', 'away_team_country_id', 'away_team_country_name', 'competition_stage_id', 'competition_stage', 'stadium_id', 'stadium', 'stadium_country_id', 'stadium_country_name', 'referee_id', 'referee', 'referee_country_id', 'referee_country_name', 'home_managers', 'away_managers', 'home_manager_id', 'home_manager_name', 'home_manager_nickname', 'home_manager_dob', 'home_manager_country_id', 'home_manager_country_name', 'away_manager_id', 'away_manager_name', 'away_manager_nickname', 'away_manager_dob', 'away_manager_country_

In [6]:
# ============================================
# LOAD SHOT EVENTS FROM ALL 35 MATCHES
# This loops through every match and extracts
# only the shot events — this is the raw
# material for our xG model
# ============================================

all_shots = []

print("Loading shot data from all matches...")

for i, match_id in enumerate(matches['match_id']):
    # Load all events for this match
    events = sb.events(match_id=match_id)
    
    # Filter to shots only
    shots = events[events['type'] == 'Shot'].copy()
    
    # Add match info
    shots['match_id'] = match_id
    
    all_shots.append(shots)
    
    # Progress update every 5 matches
    if (i + 1) % 5 == 0:
        print(f"  Loaded {i + 1}/35 matches...")

# Combine all shots into one DataFrame
shots_df = pd.concat(all_shots, ignore_index=True)

print(f"\nTotal shots loaded: {len(shots_df)}")
print(f"Columns available: {shots_df.shape[1]}")
print(f"\nShot outcomes:")
print(shots_df['shot_outcome'].value_counts())

Loading shot data from all matches...


C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


  Loaded 5/35 matches...


C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


  Loaded 10/35 matches...


...

  Loaded 25/35 matches...


C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


  Loaded 30/35 matches...


C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


  Loaded 35/35 matches...

Total shots loaded: 839
Columns available: 111

Shot outcomes:
shot_outcome
Off T               256
Saved               227
Blocked             203
Goal                111
Wayward              20
Post                 15
Saved Off Target      4
Saved to Post         3
Name: count, dtype: int64


In [7]:
# ============================================
# FIRST ANALYTICAL OBSERVATION
# ============================================

total_shots = len(shots_df)
total_goals = len(shots_df[shots_df['shot_outcome'] == 'Goal'])
conversion_rate = total_goals / total_shots * 100

observation = f"""
FIRST LOOK — La Liga 2020/2021 Shot Data
=========================================
Total matches analysed:  35
Total shots recorded:    {total_shots}
Total goals scored:      {total_goals}
Raw conversion rate:     {conversion_rate:.1f}%

SHOT OUTCOMES:
- Off Target:  256 (30.5%) — missed completely
- Saved:       227 (27.1%) — goalkeeper stopped it
- Blocked:     203 (24.2%) — outfield player blocked
- Goal:        111 (13.2%) — resulted in a goal
- Other:        42  (5.0%) — post, wayward, etc.

KEY INSIGHT:
Only 1 in 8 shots results in a goal on average.
But this average hides huge variation in shot quality.
A header from a corner is very different from a 
one-on-one with the goalkeeper.

OUR xG MODEL GOAL:
Assign each shot a probability between 0 and 1
reflecting its true likelihood of resulting in a goal
based on WHERE it was taken and HOW it was taken.
"""
print(observation)


FIRST LOOK — La Liga 2020/2021 Shot Data
Total matches analysed:  35
Total shots recorded:    839
Total goals scored:      111
Raw conversion rate:     13.2%

SHOT OUTCOMES:
- Off Target:  256 (30.5%) — missed completely
- Saved:       227 (27.1%) — goalkeeper stopped it
- Blocked:     203 (24.2%) — outfield player blocked
- Goal:        111 (13.2%) — resulted in a goal
- Other:        42  (5.0%) — post, wayward, etc.

KEY INSIGHT:
Only 1 in 8 shots results in a goal on average.
But this average hides huge variation in shot quality.
A header from a corner is very different from a 
one-on-one with the goalkeeper.

OUR xG MODEL GOAL:
Assign each shot a probability between 0 and 1
reflecting its true likelihood of resulting in a goal
based on WHERE it was taken and HOW it was taken.

